In [1]:
import pandas as pd
from sklearn.model_selection import train_test_split,GridSearchCV
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import classification_report
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OrdinalEncoder,OneHotEncoder
from sklearn.compose import ColumnTransformer

In [2]:
df = pd.read_csv(r"C:\Users\chara\desktop\Machine_learning\Logistic_Regression\loan_data.csv")

In [3]:
df.head()

,person_age,person_gender,person_education,person_income,person_emp_exp,person_home_ownership,loan_amnt,loan_intent,loan_int_rate,loan_percent_income,cb_person_cred_hist_length,credit_score,previous_loan_defaults_on_file,loan_status
0,22.0,female,Master,71948.0,0,RENT,35000.0,PERSONAL,16.02,0.49,3.0,561,No,1
1,21.0,female,High School,12282.0,0,OWN,1000.0,EDUCATION,11.14,0.08,2.0,504,Yes,0
2,25.0,female,High School,12438.0,3,MORTGAGE,5500.0,MEDICAL,12.87,0.44,3.0,635,No,1
3,23.0,female,Bachelor,79753.0,0,RENT,35000.0,MEDICAL,15.23,0.44,2.0,675,No,1
4,24.0,male,Master,66135.0,1,RENT,35000.0,MEDICAL,14.27,0.53,4.0,586,No,1


In [4]:
x=df.drop(columns='loan_status')
y=df.loan_status

In [5]:
xtrain,xtest,ytrain,ytest = train_test_split(x,y,random_state=42,train_size=0.2)

In [6]:
obj_col=x.select_dtypes(include='string').columns

In [7]:
x[obj_col].nunique()

Series([], dtype: float64)

In [8]:
x['person_education'].unique()

array(['Master', 'High School', 'Bachelor', 'Associate', 'Doctorate'],
      dtype=object)

In [9]:
order= ['High School','Associate','Bachelor','Master','Doctorate']

In [10]:
preprocessing=ColumnTransformer(
    transformers=[
        ('OneHotEncoder',OneHotEncoder(handle_unknown="ignore"),
                ['person_gender','person_home_ownership','loan_intent','previous_loan_defaults_on_file']),
        ('ordinal',OrdinalEncoder(categories=[order],handle_unknown='use_encoded_value',unknown_value=-1),
                ['person_education'])
    ],remainder="passthrough"
)

In [11]:
pipeline = Pipeline(
    steps=[("preprocessing",preprocessing),
          ('model',DecisionTreeClassifier(random_state=42))
          ]
        )

In [ ]:
grid_search_cv=GridSearchCV(
    estimator=pipeline,
    param_grid={
        'model__criterion':['gini','entropy'],
        'model__max_depth':[None,5,10],
        'model__min_samples_split':[2,5],
        'model__min_samples_leaf':[1,3],
        'model__splitter':['best','random']
    },
    verbose=10,
    n_jobs=-1 
)

In [23]:
grid_search_cv.fit(xtrain,ytrain)

Fitting 5 folds for each of 48 candidates, totalling 240 fits


,"estimator estimator: estimator objectThis is assumed to implement the scikit-learn estimator interface.Either estimator needs to provide a ``score`` function,or ``scoring`` must be passed.",Pipeline(step...m_state=42))])
,"param_grid param_grid: dict or list of dictionariesDictionary with parameters names (`str`) as keys and lists ofparameter settings to try as values, or a list of suchdictionaries, in which case the grids spanned by each dictionaryin the list are explored. This enables searching over any sequenceof parameter settings.","{'model__criterion': ['gini', 'entropy'], 'model__max_depth': [None, 5, ...], 'model__min_samples_leaf': [1, 3], 'model__min_samples_split': [2, 5], ...}"
,"scoring scoring: str, callable, list, tuple or dict, default=NoneStrategy to evaluate the performance of the cross-validated model onthe test set.If `scoring` represents a single score, one can use:- a single string (see :ref:`scoring_string_names`);- a callable (see :ref:`scoring_callable`) that returns a single value;- `None`, the `estimator`'s :ref:`default evaluation criterion ` is used.If `scoring` represents multiple scores, one can use:- a list or tuple of unique strings;- a callable returning a dictionary where the keys are the metric names and the values are the metric scores;- a dictionary with metric names as keys and callables as values.See :ref:`multimetric_grid_search` for an example.",None
,"n_jobs n_jobs: int, default=NoneNumber of jobs to run in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary `for more details... versionchanged:: v0.20 `n_jobs` default changed from 1 to None",-1
,"refit refit: bool, str, or callable, default=TrueRefit an estimator using the best found parameters on the wholedataset.For multiple metric evaluation, this needs to be a `str` denoting thescorer that would be used to find the best parameters for refittingthe estimator at the end.Where there are considerations other than maximum score inchoosing a best estimator, ``refit`` can be set to a function whichreturns the selected ``best_index_`` given ``cv_results_``. In thatcase, the ``best_estimator_`` and ``best_params_`` will be setaccording to the returned ``best_index_`` while the ``best_score_``attribute will not be available.The refitted estimator is made available at the ``best_estimator_``attribute and permits using ``predict`` directly on this``GridSearchCV`` instance.Also for multiple metric evaluation, the attributes ``best_index_``,``best_score_`` and ``best_params_`` will only be available if``refit`` is set and all of them will be determined w.r.t this specificscorer.See ``scoring`` parameter to know more about multiple metricevaluation.See :ref:`sphx_glr_auto_examples_model_selection_plot_grid_search_digits.py`to see how to design a custom selection strategy using a callablevia `refit`.See :ref:`this example`for an example of how to use ``refit=callable`` to balance modelcomplexity and cross-validated score... versionchanged:: 0.20 Support for callable added.",True
,"cv cv: int, cross-validation generator or an iterable, default=NoneDetermines the cross-validation splitting strategy.Possible inputs for cv are:- None, to use the default 5-fold cross validation,- integer, to specify the number of folds in a `(Stratified)KFold`,- :term:`CV splitter`,- An iterable yielding (train, test) splits as arrays of indices.For integer/None inputs, if the estimator is a classifier and ``y`` iseither binary or multiclass, :class:`StratifiedKFold` is used. In allother cases, :class:`KFold` is used. These splitters are instantiatedwith `shuffle=False` so the splits will be the same across calls.Refer :ref:`User Guide ` for the variouscross-validation strategies that can be used here... versionchanged:: 0.22 ``cv`` default value if None changed from 3-fold to 5-fold.",None
,"verbose verbose: intControls the verbosity: the higher, the more messages.- >1 : the computation

In [14]:
grid_search_cv.best_estimator_ # It will return the best model/estimator

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators ` for more details.","[('preprocessing', ...), ('model', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing `.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('OneHotEncoder', ...), ('ordinal', ...)]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining columns thatwere not specified in `transformers`, but present in the data passedto `fit` will be automatically passed through. This subset of columnsis concatenated with the output of the transformers. For dataframes,extra columns not seen during `fit` will be excluded from the outputof `transform`.By setting ``remainder`` to be an estimator, the remainingnon-specified columns will use the ``remainder`` estimator. Theestimator must support :term:`fit` and :term:`transform`.Note that using this feature requires that the DataFrame columnsinput at :term:`fit` and :term:`transform` have identical order.",'passthrough'
,"sparse_threshold sparse_threshold: float, default=0.3If the output of the dif

In [15]:
grid_search_cv.best_params_ # it will return the best parameter values

{'model__criterion': 'gini',
 'model__max_depth': 10,
 'model__min_samples_leaf': 3,
 'model__min_samples_split': 2,
 'model__splitter': 'best'}

In [16]:
grid_search_cv.cv_results_

{'mean_fit_time': array([0.05357327, 0.02956514, 0.05426731, 0.02990522, 0.05295277,
        0.03192191, 0.05273137, 0.04025979, 0.03546958, 0.02164125,
        0.02883911, 0.01763306, 0.02915392, 0.01815553, 0.02505746,
        0.01647444, 0.03505435, 0.01927037, 0.03207183, 0.01832547,
        0.03248925, 0.01916118, 0.03535995, 0.01930552, 0.04703712,
        0.02328553, 0.04522271, 0.02314334, 0.04578085, 0.02083664,
        0.04964819, 0.02457776, 0.02925668, 0.01766787, 0.02773829,
        0.01755853, 0.02996397, 0.01691604, 0.02616405, 0.01660976,
        0.03645539, 0.01944208, 0.03561273, 0.018642  , 0.03532195,
        0.01914349, 0.04550333, 0.01883984]),
 'std_fit_time': array([0.00508031, 0.00026248, 0.00166842, 0.00063614, 0.00193289,
        0.0029887 , 0.00351672, 0.0040856 , 0.00358025, 0.00167338,
        0.00110456, 0.00054021, 0.00686043, 0.00133383, 0.00065849,
        0.00052688, 0.00320687, 0.00174117, 0.00057491, 0.00029472,
        0.00056735, 0.00060148, 0.002

In [17]:
results = pd.DataFrame(grid_search_cv.cv_results_) 

In [18]:
# results

In [19]:
results.sort_values(by='rank_test_score')

,mean_fit_time,std_fit_time,mean_score_time,std_score_time,param_model__criterion,param_model__max_depth,param_model__min_samples_leaf,param_model__min_samples_split,param_model__splitter,params,split0_test_score,split1_test_score,split2_test_score,split3_test_score,split4_test_score,mean_test_score,std_test_score,rank_test_score
20,0.032489,0.000567,0.006190,0.000733,gini,10,3,2,best,"{'model__criterion': 'gini', 'model__max_depth...",0.915000,0.913333,0.916667,0.920556,0.906667,0.914444,0.004568,1
22,0.035360,0.002494,0.007221,0.001169,gini,10,3,5,best,"{'model__criterion': 'gini', 'model__max_depth...",0.915000,0.913333,0.916667,0.920556,0.906667,0.914444,0.004568,1
18,0.032072,0.000575,0.005885,0.000290,gini,10,1,5,best,"{'model__criterion': 'gini', 'model__max_depth...",0.913889,0.911667,0.914444,0.922222,0.909444,0.914333,0.004323,3
40,0.036455,0.000779,0.006056,0.000531,entropy,10,1,2,best,"{'model__criterion': 'entropy', 'model__max_de...",0.908333,0.913889,0.912778,0.922778,0.913333,0.914222,0.004709,4
16,0.035054,0.003207,0.006967,0.000711,gini,10,1,2,best,"{'model__criterion': 'gini', 'model__max_depth...",0.911111,0.913333,0.913889,0.921667,0.910556,0.914111,0.003985,5
44,0.035322,0.000596,0.006225,0.000438,entropy,10,3,2,best,"{'model__criterion': 'entropy', 'model__max_de...",0.908333,0.909444,0.915556,0.921667,0.912778,0.913556,0.004787,6
46,0.045503,0.017199,0.006007,0.000529,entropy,10,3,5,best,"{'model__criterion': 'entropy', 'model__max_de...",0.908333,0.909444,0.915556,0.921667,0.912778,0.913556,0.004787,6
12,0.029154,0.006860,0.005571,0.000198,gini,5,3,2,best,"{'model__criterion': 'gini', 'model__max_depth...",0.908333,0.908889,0.917222,0.918889,0.912222,0.913111,0.004283,8
42,0.035613,0.000222,0.005868,0.000300,entropy,10,1,5,best,"{'model__criterion': 'entropy', 'model__max_de...",0.907778,0.910000,0.911667,0.923333,0.912778,0.913111,0.005382,8
14,0.025057,0.000658,0.006403,0.000813,gini,5,3,5,best,"{'model__criterion': 'gini', 'model__max_depth...",0.908333,0.908889,0.917222,0.918889,0.912222,0.913111,0.004283,8


In [24]:
grid_search_cv.predict(xtrain)

array([1, 0, 0, ..., 0, 1, 0], shape=(9000,))